<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>

<p><font size="5" color='grey'> <b>
DeepAgents — Multi-Skill Progressive Disclosure
</b></font> </br></p>

---

**Beitrag zum Leitprojekt:** Multi-Skill-Routing zeigt, wie ein Agent zwischen mehreren spezialisierten Skills — Freigabeprüfung (Compliance), Dokumentenrecherche und PMO-Briefing — entscheidet, welcher Skill zur Anfrage passt. Alle drei Skills erweitern denselben PMO-Assistenten: vorbereiten, belegen und freigeben.

Das Modul zeigt, wie der Agent bei einer Anfrage den passenden Skill auswählt oder bei Überschneidungen mehrere kombiniert.


**Praxis-Transfer:** Mara Vogt profitiert von mehreren Skills, wenn ein Briefing gleichzeitig Aktionspunkte, Compliance-Hinweise und Quellenrelevanz berücksichtigen muss.


In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import json
import os
from pathlib import Path

os.environ["LANGSMITH_TRACING"] = "false"
from langsmith import tracing_context
# Tracing bleibt standardmäßig aus; ausgewählte Referenzläufe werden explizit aktiviert.
os.environ["LANGSMITH_PROJECT"]  = "m35-multi-skill"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace,
    copy_from_github,
)

setup_api_keys(["OPENAI_API_KEY", "LANGSMITH_API_KEY"], create_globals=False)
print()
check_environment()
print()
get_ipinfo()
from genai_lib.model_config import (
    BASELINE,
    ROUTER,
    JUDGE,
    PLANNER,
    WORKER,
    WORKER_PREMIUM,
    CODING,
    EMBEDDINGS,
)
# LangSmith Tracing
run_cfg = {
    "tags": ["m35", "deepagents"],
    "metadata": {"notebook": "m35", "version": "1.0"}
}


In [ ]:
#@title 📦 Installationen { display-mode: "form" }

# Getestete Versionen (Stand: März 2026)
# deepagents 0.4.x: skills=[...]-Parameter ist stabil ab v0.4.10
# ⚠️ Vor einem Update: CHANGELOG prüfen — Sub-Agent-Dict-Syntax kann sich ändern
!uv pip install --system -q "deepagents==0.4.12"

# 1 | Überblick
---

<font color='black' size='5'>Von einem Skill zu vielen</font>

DeepAgents scannt ein Skills-Verzeichnis beim Start, liest die `description`-Felder aus allen `SKILL.md`-Dateien und entscheidet
bei jeder Anfrage dynamisch, welcher Skill passt — **Progressive Disclosure** (schrittweise Offenlegung von Informationen).

| Lernziel | Beschreibung |
|----------|--------------|
| Skills-Scan | Agent liest SKILL.md-Frontmatter automatisch beim Start |
| Skill-Routing | Dynamische Auswahl anhand von Trigger-Phrasen |
| GitHub-Cache | Skill-Dateien aus GitHub laden, lokal cachen, versionieren |
| Abgrenzung | Vergleich manueller vs. nativer Skill-Ansatz |


<p><font color='black' size="5">
Architekturmuster: Skills + Subagents kombinieren
</font></p>

Dieses Modul führt das wichtigste Architekturmuster im DeepAgents-Ökosystem ein:
**Skills liefern das Wissen, Subagents führen die Arbeit isoliert aus.**

Die Rollenverteilung:

| Ebene | Aufgaben | Beispiel in diesem Modul |
|-------|---------|--------------------------|
| **Skill** | Definiert *wie* eine Aufgaben angegangen wird — Ablauf, Regeln, Referenzen | `pmo-briefing/SKILL.md`, `compliance/SKILL.md` |
| **Koordinator-Agent** | Erkennt welcher Skill passt, leitet an Subagent weiter | gpt-5.6-luna — entscheidet über Routing |
| **Sub-Agent** | Führt die Skill-gesteuerte Aufgaben in *isolierter History* aus | `briefing-writer` (gpt-5.4-mini) |

**Warum Isolation wichtig ist:**

Ohne Subagent-Isolation würde jede Skill-Ausführung die Message-History des Koordinators aufblähen. Bei mehreren Skills und langen Kontext-Dokumenten entsteht schnell Kontext-Overflow. Ein Sub-Agent bekommt nur das, was er für seine Aufgaben braucht.

**Warum Skills und Subagents sich ergänzen:**

Ein Skill ohne Subagent ist möglich — der Koordinator führt dann selbst aus. Ein Subagent ohne Skill ist ebenfalls möglich — er erhält dann nur den `system_prompt`.
Die Kombination entfaltet ihren Mehrwert, wenn:

1. Die Skill-Logik **zu komplex ist für den Koordinator** (viele Schritte, Referenzen, spezifisches Format)
2. Der Koordinator **mehrere Subagenten** für verschiedene Skills hat (dieses Modul)
3. Die Subagenten **unterschiedliche Modelle** benötigen (seit 0.4.11: `model=` im SubAgent-Dict)

```
Anfrage → Koordinator (Skill-Routing) → Sub-Agent A (Skill X ausführen)
                                       → Sub-Agent B (Skill Y ausführen)
```


In [ ]:
#@markdown   <p><font size="4" color="green">Architektur-Übersicht</font></p>

diagram_arch = '''
%%{init: {
  'theme': 'base',
  'themeVariables': {
    'primaryColor': '#e0f2fe',
    'primaryBorderColor': '#0284c7',
    'primaryTextColor': '#1e293b',
    'lineColor': '#64748b',
    'clusterBkg': '#f8fafc',
    'clusterBorder': '#cbd5e1',
    'noteBkgColor': '#fef3c7',
    'noteBorderColor': '#d97706',
    'noteTextColor': '#78350f',
    'edgeLabelBackground': '#ffffff'
  }
}}%%
flowchart TB
    GH([" 🐙 GitHub Repo\nralf-42/Agenten"]) -->|"copy_from_github()"| CACHE

    subgraph CACHE [" 📁 Lokaler Skills-Cache"]
        direction TB
        S1["compliance/\nSKILL.md · references/ · scripts/"]
        S2["pmo-briefing/\nSKILL.md · references/ · scripts/"]
        S3["dokumentenrecherche/\nSKILL.md · references/ · scripts/"]
    end

    CACHE -->|"skills=[cache_dir]"| AGENT

    subgraph AGENT [" 🤖 DeepAgent Koordinator — gpt-5.6-sol"]
        SCAN["Skill-Scan\n(Frontmatter lesen)"] --> INJECT["descriptions →\nSystem-Prompt"]
        INJECT --> ROUTE["Anfrage analysieren\nSkill auswählen"]
    end

    U([" 🧑 Nutzeranfrage"]) --> ROUTE
    ROUTE -->|"compliance"| R1[" ✅ Freigabe-Workflow"]
    ROUTE -->|"meeting"| R2[" 📋 Briefing-Workflow"]
    ROUTE -->|"dokumentenrecherche"| R3[" 🔍 Dokumentenrecherche"]

    %% Farbdefinitionen nach Funktionsgruppe
    classDef default fill:#e0f2fe,stroke:#0284c7,stroke-width:2px,color:#0c4a6e;
    classDef input fill:#e0e7ff,stroke:#6366f1,stroke-width:2px,color:#1e1b4b;
    classDef metric fill:#fef3c7,stroke:#d97706,stroke-width:2px,color:#78350f;
    classDef output fill:#dcfce7,stroke:#16a34a,stroke-width:2px,color:#14532d;
    classDef target fill:#fae8ff,stroke:#c026d3,stroke-width:2px,color:#701a75;

    %% Zuweisung
    class GH,U input;
    class ROUTE metric;
    class R1,R2,R3 output;
    class S1,S2,S3 target;
'''
mermaid(diagram_arch, width=700)

# 2 | Setup

---

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends.utils import create_file_data
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from pydantic import BaseModel, Field

from genai_lib.model_config import ROUTER, WORKER  # gpt-5.6-luna, gpt-5.4-mini

# Koordinator: gpt-5.6-luna (Demo-Routing, kein temperature)
# gpt-5.6-sol wäre zu scharf für diesen Demo-Kontext und hat engeres TPM-Limit (30k)
# Mit Tools/Skills: gpt-5.6-luna verlangt reasoning_effort="none"
coord_llm = init_chat_model(ROUTER, reasoning_effort="none")

mprint("✅ Pakete geladen")

# 3 | GitHub-Skills laden und lokal cachen

---

Alle `SKILL.md`-Dateien werden aus dem GitHub-Repo `ralf-42/Agenten` geladen
und lokal in `./_skills_cache/` gespeichert.

**Warum lokaler Cache statt direkter GitHub-URL?**

| Aspekt | Direkte URL | Lokaler Cache |
|--------|-------------|---------------|
| Reproduzierbarkeit | ❌ `main` kann sich ändern | ✅ Commit-SHA pinnen |
| Offline-Betrieb | ❌ | ✅ |
| `skills=[...]`-API | ❌ (erwartet lokales FS) | ✅ |
| GitHub-Rate-Limit | ⚠️ bei vielen Runs | ✅ einmal laden |

> 💡 **Commit-SHA pinnen** für vollständige Reproduzierbarkeit:
> `COMMIT_SHA = "abc1234"` statt `"main"`

In [ ]:
# 3.1 Skill-Verzeichnisse komplett aus GitHub laden und lokal cachen

CACHE_DIR = Path("./06_skill")

copy_from_github(
    source="ralf-42/Agenten/06_skill",
    target=str(CACHE_DIR),
)

# Alle Skill-Verzeichnisse auto-entdecken: jedes direkte Unterverzeichnis mit SKILL.md
cached_files = {
    p.parent.name: p
    for p in sorted(CACHE_DIR.rglob("SKILL.md"))
    if p.parent.parent == CACHE_DIR
}
for skill_name, p in cached_files.items():
    mprint(f"✅ `{skill_name}` → `{p.parent}/`")

# 4 | Lokales Skills-Verzeichnis inspizieren

---

Bevor der Agent gebaut wird, zeigen wir explizit, was er beim Start sieht:
die `name`- und `description`-Felder aus dem YAML-Frontmatter jeder `SKILL.md`.

Genau diese `description`-Werte werden in den System-Prompt injiziert,
sodass der Agent bei einer Anfrage erkennt, welcher Skill relevant ist.

In [ ]:
# 4.1 Frontmatter aus SKILL.md-Dateien lesen und anzeigen

import re
import yaml

def read_frontmatter(path: Path) -> dict:
    """Liest YAML-Frontmatter aus einer Markdown-Datei (unterstützt Block-Scalars)."""
    text  = path.read_text(encoding="utf-8")
    match = re.match(r"^---(.+?)---", text, re.DOTALL)
    if not match:
        return {}
    return yaml.safe_load(match.group(1)) or {}

zeilen = ["## 📁 Skills-Verzeichnis: was der Agent beim Start sieht", ""]
for skill_name, p in cached_files.items():
    fm   = read_frontmatter(p)
    name = fm.get("name", "—")
    desc = (fm.get("description", "—") or "—")
    zeilen += [
        f"**`{skill_name}`** &nbsp;·&nbsp; name: `{name}`",
        f"> {desc}",
        "",
    ]
mprint("\n".join(zeilen))

# 5 | DeepAgent mit skills=[...] erzeugen

---

Die `skills=[...]`-API erwartet **lokale Verzeichnispfade**. Der Agent scannt
beim ersten Invoke alle Unterordner nach `SKILL.md`-Dateien und injiziert
deren `description`-Felder in den System-Prompt.

**Wichtig:** `skills_files` übergibt den Datei-Inhalt an das virtuelle Dateisystem
des Agents. Der Pfad-Prefix `/skills/` im Dict muss mit dem `skills=["/skills/"]`-
Parameter übereinstimmen.

```python
**Minimales Muster**
agent = create_deep_agent(
    model=llm,
    skills=["/skills/"],   # virtueller FS-Pfad
    checkpointer=InMemorySaver(),
)
result = agent.invoke(
    {"messages": [...], "files": skills_files},
    config={"configurable": {"thread_id": "..."}, "recursion_limit": 50},
)
```

In [ ]:
# 5.1 Einfache Mock-Tools (Demo-Zwecke: zeigt Skill-Routing, nicht volle Logik)

@tool
def compliance_check(subject_name: str, recipient_scope: str, data_basis: str = "testkorpus") -> str:
    """
    Berechnet den Freigabe-Risikoscore für ein Briefing oder einen Bericht.
    Gibt low, medium oder high zurück.
    """
    # Demo: einfache Heuristik — im Skill übernimmt scripts/assess_risk.py die Bewertung
    extern = recipient_scope.lower() == "extern"
    produktiv = data_basis.lower() == "produktivdaten"
    risk = "high" if (extern and produktiv) else ("medium" if (extern or produktiv) else "low")
    return json.dumps({"risk_level": risk, "subject": subject_name, "recipient_scope": recipient_scope, "data_basis": data_basis})


@tool
def aktionspunkte_extrahieren(text: str) -> str:
    """
    Extrahiert Action Items aus Meeting-Text deterministisch.
    Gibt JSON-Liste zurück.
    """
    # Demo: Schlüsselwörter-Heuristik
    actions = []
    keywords = ["bis", "bis zum", "deadline", "verantwortlich", "action"]
    for line in text.split("\n"):
        if any(kw in line.lower() for kw in keywords):
            actions.append({"task": line.strip()[:80], "owner": "[offen]", "due": "[offen]"})
    if not actions:
        actions = [{"task": "Keine Action Items gefunden", "owner": "—", "due": "—"}]
    return json.dumps(actions, ensure_ascii=False)


@tool
def assess_source_relevance(source_text: str, query: str) -> str:
    """
    Bewertet die Relevanz einer Quelle für eine Suchanfrage.
    Gibt Score zwischen 0.0 und 1.0 zurück.
    """
    # Demo: Überlappung von Wörtern als Relevanz-Proxy
    query_words = set(query.lower().split())
    source_words = set(source_text.lower().split())
    overlap = len(query_words & source_words)
    score = min(1.0, overlap / max(len(query_words), 1) * 1.5)
    return json.dumps({"score": round(score, 2), "query": query})


TOOLS = [compliance_check, aktionspunkte_extrahieren, assess_source_relevance]
mprint(f"✅ {len(TOOLS)} Mock-Tools registriert")

In [ ]:
# 5.2 skills_files-Dict aufbauen (virtuelles Dateisystem — alle Skill-Dateien)

VIRT_PREFIX = "/skills/"  # muss mit skills=[VIRT_PREFIX] übereinstimmen

skills_files = {}
zeilen = []
for skill_name, skill_md_path in cached_files.items():
    skill_dir = skill_md_path.parent
    for fpath in sorted(skill_dir.rglob("*")):
        if fpath.is_file():
            rel       = fpath.relative_to(CACHE_DIR)
            virt_path = f"{VIRT_PREFIX}{rel}"
            skills_files[virt_path] = create_file_data(fpath.read_text(encoding="utf-8"))
            zeilen.append(f"📄 `{virt_path}`")

zeilen.append(f"\n✅ **{len(skills_files)} Dateien** im virtuellen FS registriert")
mprint("\n".join(zeilen))

In [ ]:
# 5.3 DeepAgent mit nativer skills=[...]-API erzeugen

import time

multi_skill_agent = create_deep_agent(
    model=coord_llm,
    tools=TOOLS,
    skills=[VIRT_PREFIX],          # ← native Skills-API
    checkpointer=InMemorySaver(),
    system_prompt=(
        "Du bist ein Allround-Assistent mit Zugriff auf spezialisierte Skills. "
        "Wähle bei jeder Anfrage den passenden Skill aus den verfügbaren Skills. "
        "Wenn kein Skill passt, antworte direkt ohne Skill. "
        "Antworte immer auf Deutsch."
    ),
)


def invoke_agent(anfrage: str, session_id: str = None) -> str:
    """Hilfsfunktion: Agent aufrufen und letzte Antwort zurückgeben."""
    sid = session_id or f"m35-{int(time.time())}"
    result = multi_skill_agent.invoke(
        {
            "messages": [HumanMessage(content=anfrage)],
            "files":    skills_files,
        },
        config={
            "configurable": {"thread_id": sid},
            "recursion_limit": 50,
            "run_name": f"m35_kap5_skillrouting_{sid.removeprefix('m35-')}",
        },
    )
    return result["messages"][-1].content


mprint("✅ Multi-Skill-Agent bereit")

**Was passiert hier?**

1. `InMemorySaver()` — speichert den Graph-Zustand im RAM für Session-Persistenz
2. `create_deep_agent(...)` — erstellt einen DeepAgent mit erweiterter Reasoning-Fähigkeit

# 6 | Demo 1 — Compliance-Anfrage

---

**Erwartetes Verhalten:** Der Agent erkennt an der Formulierung
("Briefing freigeben", "Freigabeprüfung", "Versand prüfen"), dass der
`compliance-skill` passt, liest die vollständige `SKILL.md` und
führt den definierten Workflow aus.

> 📌 Progressive Disclosure: Erst die `description` wird gelesen.
> Erst bei einem Match liest der Agent die volle SKILL.md.

In [ ]:
# Demo 1: Compliance — Freigabeprüfung eines Statusberichts

anfrage_compliance = """
Ich möchte den Statusbericht zu Projekt Kompass (KW 21) an einen externen Dienstleister senden.
Er enthält vertrauliche Budgetangaben und nutzt produktive Daten.
Die Datenschutzprüfung wurde durchgeführt: Die Freigabe gilt nur für den kuratierten Testkorpus.
Bitte führe die Freigabeprüfung durch und gib eine Empfehlung.
"""

mprint(f"**📋 Anfrage:** {anfrage_compliance.strip()}")

antwort_1 = invoke_agent(anfrage_compliance, session_id="demo1-compliance")

mprint(f"**🤖 Agent-Antwort:** {antwort_1}")

# 7 | Demo 2 — PMO-Briefing-Anfrage

---

**Erwartetes Verhalten:** Trigger-Phrasen wie "Meeting vorbereiten",
"Agenda", "Briefing" matchen den technischen Skill „pmo-briefing“.
Der Skill-Workflow strukturiert die Agenda und extrahiert Action Items.

In [ ]:
with tracing_context(enabled=True):
    # Demo 2: PMO-Briefing — nächsten Steuerkreis vorbereiten

    anfrage_meeting = """
    Bereite den nächsten Steuerkreis vor.
    Teilnehmer: Mara Vogt, Jonas Reiter, Elif Kaya, Dr. Thomas Brandt, Sabine Holler.
    Themen: Rechte- und Rollensteuerung (vertagt), Stand von Risiko R2 (mitigiert),
    Entscheidung D2 zur Vektordatenbank.
    Offene Action Items aus dem letzten Meeting: Mara Vogt bereitet die Entscheidung zur
    Rechte- und Rollensteuerung vor, Jonas Reiter bereitet Logging- und Zugriffskonzepte vor.
    """

    mprint(f"**📋 Anfrage:** {anfrage_meeting.strip()}")

    antwort_2 = invoke_agent(anfrage_meeting, session_id="demo2-pmo-briefing")

    mprint(f"**🤖 Agent-Antwort:** {antwort_2}")

# 8 | Demo 3 — Mehrdeutige Anfrage mit Skill-Auswahl

---

**Erwartetes Verhalten:** Die Anfrage berührt mehrere Skill-Bereiche.
Der Agent muss entscheiden: Recherche-Skill? Compliance-Skill? Kombination?

Dieses Demo zeigt die **Grenzen und Stärken** des automatischen Routings:
- Bei klaren Trigger-Phrasen → eindeutiger Match
- Bei Überschneidungen → Agent priorisiert oder kombiniert
- Bei keinem Match → direkter Antwort ohne Skill

In [ ]:
with tracing_context(enabled=True):
    # Demo 3: Mehrdeutige Anfrage — Recherche + Compliance-Bezug

    anfrage_mixed = """
    Ich bereite den Steuerkreis zur Rechte- und Rollensteuerung vor.
    Kannst du mir die relevanten Hintergrundinformationen aus den Projektunterlagen zusammenstellen
    und gleichzeitig prüfen, ob der Briefing-Entwurf an externe Empfänger
    freigegeben werden darf?
    """

    mprint(f"**📋 Anfrage:** {anfrage_mixed.strip()}")

    antwort_3 = invoke_agent(anfrage_mixed, session_id="demo3-recherche-compliance")

    mprint(f"**🤖 Agent-Antwort:** {antwort_3}")

<p><font color='black' size="5">
🔍 Skill-Routing-Analyse für Demo 3
</font></p>


| Skill | Trigger-Signal in Anfrage | Erwarteter Match |
|-------|--------------------------|------------------|
| compliance | „freigegeben werden darf“, „externe Empfänger“ | ⚠️ Teilmatch |
| pmo-briefing | „Steuerkreis vorbereiten“, „Briefing-Entwurf“ | ⚠️ Teilmatch |
| dokumentenrecherche | „Hintergrundinformationen“, „Projektunterlagen“ | ✅ Starker Match |

> 💡 Bei Überschneidungen wählt der Agent den stärksten Match oder kombiniert Skills — je nach LLM-Entscheidung. Das Verhalten ist nicht deterministisch und kann variieren.

# 9 | Vergleich: natives Skill-System vs. manueller Ansatz

---

Beide Ansätze sind didaktisch wertvoll und ergänzen sich:

| Dimension | Manuell (Gate-Writer) | Nativ (Progressive Disclosure) |
|-----------|--------------------------|--------------------------------------|
| **Skill-Loading** | Explizit via `urllib.request` | Automatisch via `skills=[...]` |
| **Skill-Quelle** | GitHub Raw URL (direkt) | GitHub → lokaler Cache |
| **Routing** | Fest verdrahtet (ein Skill) | Dynamisch durch LLM |
| **Anzahl Skills** | 1 (pmo-briefing) | 3 (compliance, pmo-briefing, dokumentenrecherche) |
| **Gate-Writer** | ✅ Vollständig (gpt-5.6-sol → gpt-5.4-mini) | ❌ Nicht implementiert |
| **Kontrolle** | ✅ Sehr hoch | ⚠️ Mittel (LLM entscheidet) |
| **Transparenz** | ✅ Jeder Schritt sichtbar | ⚠️ Routing ist black-box |
| **Erweiterbarkeit** | Aufwendig (neuer Code) | ✅ Neuer Ordner + SKILL.md |
| **Reproduzierbarkeit** | ✅ Hoch (SHA-pinned) | ✅ Hoch (SHA-pinned + Version) |

**Wann welchen Ansatz wählen?**

```
Genau ein Skill, kritische Logik, Gate-Writer-Qualität   → Manueller Ansatz
Mehrere Skills, flexible Erweiterung, Rapid Prototyping  → Nativer Ansatz
Produktion mit Audit-Trail                               → Manueller Ansatz
Interne Tools / Wissensdatenbank                         → Nativer Ansatz
```

In [ ]:
#@markdown   <p><font size="4" color="green">Entscheidungsbaum: Manuell vs. Nativ</font></p>

diagram_compare = '''
%%{init: {
  'theme': 'base',
  'themeVariables': {
    'primaryColor': '#e0f2fe',
    'primaryBorderColor': '#0284c7',
    'primaryTextColor': '#1e293b',
    'lineColor': '#64748b',
    'clusterBkg': '#f8fafc',
    'clusterBorder': '#cbd5e1',
    'noteBkgColor': '#fef3c7',
    'noteBorderColor': '#d97706',
    'noteTextColor': '#78350f',
    'edgeLabelBackground': '#ffffff'
  }
}}%%
flowchart TB
    START([" ❓ Neues Skill-System bauen"]) --> Q1{Wie viele Skills?}

    Q1 -->|"genau 1"| Q2{Gate-Writer
Qualität nötig?}
    Q1 -->|"> 1"| Q3{Routing flexibel
bzw. erweiterbar?}

    Q2 -->|"ja"| M35[" 🔵 Manueller Ansatz\nManuelle Kontrolle"]
    Q2 -->|"nein"| M34A[" 🟢 Nativer Ansatz\nEinfacher"]

    Q3 -->|"ja"| M34B[" 🟢 Nativer Ansatz\nProgressive Disclosure"]
    Q3 -->|"nein"| Q4{Audit-Trail
erforderlich?}

    Q4 -->|"ja"| M35
    Q4 -->|"nein"| M34B

    %% Farbdefinitionen nach Funktionsgruppe
    classDef default fill:#e0f2fe,stroke:#0284c7,stroke-width:2px,color:#0c4a6e;
    classDef input fill:#e0e7ff,stroke:#6366f1,stroke-width:2px,color:#1e1b4b;
    classDef metric fill:#fef3c7,stroke:#d97706,stroke-width:2px,color:#78350f;
    classDef output fill:#dcfce7,stroke:#16a34a,stroke-width:2px,color:#14532d;

    %% Zuweisung
    class START input;
    class Q1,Q2,Q3,Q4 metric;
    class M35,M34A,M34B output;
'''
mermaid(diagram_compare, width=550)

# 10 | Grenzen, Stabilität, didaktische Einordnung

---

## Bekannte Grenzen

| Grenze | Details |
|--------|---------|
| **Lokales FS erforderlich** | `skills=[...]` akzeptiert keine GitHub-URLs direkt — Cache-Schritt ist immer nötig |
| **Nicht deterministisch** | LLM-Routing kann bei ähnlichen Trigger-Phrasen variieren |
| **Kein Gate-Writer** | Natives Skill-System hat keine eingebaute zweistufige Analyse |
| **Versions-API** | `skills=`-Parameter ist in v0.4 noch nicht als stable markiert |
| **Debugging** | Schwieriger als beim manuellen Ansatz: Welcher Skill gewählt wurde, ist nicht direkt sichtbar |

## Stabilität (Stand: März 2026)

```python
**Getestete, stabile Kombination**
deepagents==0.4.12
langchain>=1.0.0
langgraph>=1.0.0
```

> ⚠️ **Vor einem Versions-Update:** `/check-deepagents`-Command ausführen
> und das CHANGELOG überprüfen.

## Einordnung

Manueller und nativer Skill-Ansatz sind **keine Alternativen**, sondern **komplementär**.
Beide zusammen ergeben das vollständige Bild moderner Skill-basierter Agenten.


# A | Aufgaben
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen; alternative Herausforderungen sind möglich.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Geeignet ist sie zum Beispiel, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


**Grundlagen**

Teste das native Skill-System mit eigenen Anfragen: Formuliere je eine Anfrage pro vorhandenem Skill und prüfe, ob der Agent sie an den passenden Skill routet.

In [ ]:
# Grundlagen: Skill-Routing testen
# Startpunkt: multi_skill_agent und invoke_agent() aus dem Kapitel "DeepAgent mit skills=[...] erzeugen"

anfragen = [
    '...',  # sollte Skill 1 auslösen
    '...',  # sollte Skill 2 auslösen
]
# for anfrage in anfragen:
#     antwort = invoke_agent(anfrage)
#     print(antwort)


**Aufbau**

Ergänze einen eigenen Skill (Verzeichnis mit `SKILL.md`) und teste, ob das Routing korrekt funktioniert: Jeder Skill wird nur bei passenden Anfragen aktiviert.

In [ ]:
# Aufbau: Eigenen Skill ergänzen + Routing testen
# Startpunkt: Kapitel "DeepAgent mit skills=[...] erzeugen" (skills_files, VIRT_PREFIX)

# 1. Neues Skill-Verzeichnis mit SKILL.md (Frontmatter: name, description) anlegen
# 2. Datei wie die vorhandenen Skills in skills_files aufnehmen (create_file_data)
# 3. Agent neu erzeugen (create_deep_agent mit skills=[VIRT_PREFIX])
# 4. testfragen-Liste mit (frage, erwarteter_skill) definieren
# 5. Für jede Frage invoke_agent() aufrufen und erwarteten mit tatsächlichem Skill vergleichen


**Vertiefung** Implementiere ein eigenes Routing-Szenario mit drei Skills.
Bei einer unklaren Anfrage darf kein Skill aufgerufen werden. Stattdessen muss
invoke_agent() eine Antwort zurückgeben, die mit Rueckfrage: beginnt und
gezielt nach der fehlenden Information fragt. Teste mindestens eine eindeutige
Anfrage pro Skill und eine vage Anfrage.


In [ ]:
# Vertiefung: Drei Skills + festgelegtes Verhalten bei vager Anfrage
# 1. Drei Skill-Verzeichnisse mit SKILL.md registrieren.
# 2. System-Prompt ergaenzen: Bei unklarer Anfrage keinen Skill aufrufen,
#    sondern mit 'Rueckfrage:' beginnen und die fehlende Information nennen.
# 3. Eindeutige Testfragen und eine vage Frage formulieren.
# 4. Pruefen: Bei der vagen Frage erscheint 'Rueckfrage:' und kein Skill-Name
#    wird als ausgewaehlter Skill ausgegeben.


<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [KI-Agenten-Architektur](https://ralf-42.github.io/Agenten/visualisierungen/ki_agent_architektur/)
- [LangGraph](https://ralf-42.github.io/Agenten/visualisierungen/langgraph_explainer/)


# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [Skills](https://ralf-42.github.io/Agenten/06-multi-agent-erweiterungen/skills.html)
- [Einsteiger DeepAgents](https://ralf-42.github.io/Agenten/06-multi-agent-erweiterungen/einsteiger-deepagents.html)
- [Multi-Agent-Systeme](https://ralf-42.github.io/Agenten/06-multi-agent-erweiterungen/multi-agent-systeme.html)
- [Agenten-Kommunikationsprotokolle](https://ralf-42.github.io/Agenten/06-multi-agent-erweiterungen/agenten-kommunikationsprotokolle.html)
- [Checkliste Agentensystem](https://ralf-42.github.io/Agenten/04-agenten-implementierung/checkliste-agentensystem.html)
